In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import lime
from lime import lime_image
import shap

# Load image dataset
digits = load_digits()
X = digits.images
y = digits.target

# Flatten images for KNN
X_flat = X.reshape((X.shape[0], -1))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)

# Train KNN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# Pick an image to explain
idx = 0
image = X_test[idx].reshape(8, 8)

# ----------- LIME for image explanation ----------- #
from skimage.color import gray2rgb
from skimage.segmentation import slic

# LIME needs images in [0, 255] range and RGB
image_rgb = gray2rgb(image) * 16  # Scale up from 0–16 to 0–255
explainer = lime_image.LimeImageExplainer()

def predict_fn(images):
    # flatten and predict with knn
    images = np.array([img[:, :, 0].reshape(-1) for img in images])  # extract one channel
    return knn.predict_proba(images)

explanation = explainer.explain_instance(
    image_rgb.astype('double'),
    classifier_fn=predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=1000,
    segmentation_fn=lambda x: slic(x, n_segments=16, compactness=1, sigma=1)
)

# Show LIME explanation
from lime.visualization import mark_boundaries
temp, mask = explanation.get_image_and_mask(
    label=explanation.top_labels[0],
    positive_only=True,
    hide_rest=False
)
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title('LIME explanation')
plt.axis('off')
plt.show()

# ----------- SHAP for KNN (tabular) ----------- #
# Use SHAP KernelExplainer, since KNN is not tree- or DL-based
explainer = shap.KernelExplainer(knn.predict_proba, shap.kmeans(X_train, 10))
shap_values = explainer.shap_values(X_test[:1])

# Visualize SHAP values
shap.image_plot([shap_values[i].reshape(1, 8, 8) for i in range(10)], images=X_test[:1].reshape(1, 8, 8))
